# DataCity Analytics
Notebook demonstrativo. Execute após popular o MongoDB, a partir da raiz do projeto.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
from config.database import conectar
from repositories.viagens_repository import consultar
from services.preparacao_dados import preparar
from services.indicadores import calcular
from visualizations.graficos import gerar

In [2]:
cliente, db = conectar()
print({nome: db[nome].count_documents({}) for nome in ["linhas", "veiculos", "viagens", "ocorrencias"]})
df, qualidade = preparar(consultar(db), list(db.veiculos.find()))
print(qualidade)
df.head()

{'linhas': 50, 'veiculos': 500, 'viagens': 20000, 'ocorrencias': 5000}
{'ausentes_antes': {'id_viagem': 0, 'data': 0, 'hora': 0, 'linha': 0, 'veiculo': 0, 'regiao': 0, 'passageiros': 1, 'duracao_minutos': 0, 'atraso_minutos': 0, 'avaliacao': 1, 'tipo_ocorrencia': 0}, 'duplicatas_logicas': 1, 'horas_invalidas': 1, 'lotacao_acima_capacidade': 703, 'passageiros_impossiveis': 1, 'duracoes_invalidas': 1, 'adiantamentos': 3472, 'linhas_apos_limpeza': 19999}


,id_viagem,data,hora,linha,veiculo,regiao,passageiros,duracao_minutos,atraso_minutos,avaliacao,tipo_ocorrencia,capacidade
0,VIA00001,2026-09-12 00:00:00+00:00,12:45,LIN050,VEI0500,Centro,34.0,51.0,12,NaN,transito,80
1,VIA00002,2026-09-26 00:00:00+00:00,20:45,LIN029,VEI0281,Norte,35.0,46.0,-2,5.0,nenhuma,45
2,VIA00003,2026-09-01 00:00:00+00:00,07:30,LIN046,VEI0458,Centro,49.0,26.0,-8,5.0,nenhuma,80
3,VIA00004,2026-09-12 00:00:00+00:00,22:15,LIN016,VEI0154,Sul,31.0,38.0,-15,5.0,nenhuma,60
4,VIA00005,2026-09-28 00:00:00+00:00,23:15,LIN044,VEI0436,Centro,NaN,35.0,6,2.0,nenhuma,80


## Qualidade
O tratamento normaliza regiões e horas, remove duplicatas lógicas da análise e verifica capacidade. Atraso negativo representa adiantamento.

In [3]:
indicadores = calcular(df)
print(indicadores["resumo"])
indicadores["regiao"].sort_values("atraso_medio",ascending=False)

{'viagens': 19999, 'passageiros_medios': 37.07896184427664, 'duracao_media': 42.175567556755674, 'atraso_medio': 7.5348767438371915, 'atraso_mediano': 7.0, 'avaliacao_media': 3.8695369536953694, 'correlacao_atraso_avaliacao': -0.6332694773135596, 'lotacao_acima_capacidade': 702}


,regiao,viagens,passageiros,atraso_medio,atraso_mediano,duracao_media,avaliacao_media
4,Sul,4015,151752.0,7.852802,8.0,42.330262,3.845081
0,Centro,4141,151967.0,7.498672,7.0,42.015697,3.869324
3,Oeste,3901,143382.0,7.471161,7.0,42.102025,3.887465
1,Leste,3960,142816.0,7.467929,7.0,42.233333,3.865404
2,Norte,3982,151551.0,7.380964,7.0,42.200452,3.880964


In [4]:
indicadores["horario"].sort_values("passageiros",ascending=False).head(10)

,hora,viagens,passageiros,atraso_medio,atraso_mediano,duracao_media,avaliacao_media
12,08:00,296,15821.0,11.543919,12.0,43.699324,3.608108
51,17:45,284,15287.0,11.366197,11.0,44.602113,3.693662
52,18:00,286,15287.0,12.548951,12.0,44.695804,3.573427
48,17:00,282,14977.0,11.106383,10.0,43.921986,3.648936
49,17:15,267,14425.0,11.397004,11.0,44.423221,3.602996
8,07:00,275,14286.0,11.978182,12.0,45.163636,3.570909
9,07:15,262,13914.0,11.484733,11.0,44.362595,3.633588
10,07:30,270,13859.0,11.925926,11.0,44.914815,3.514815
15,08:45,253,13841.0,11.173913,10.0,42.992095,3.620553
55,18:45,265,13692.0,10.826415,11.0,43.230189,3.679245


In [5]:
gerar(df, indicadores)
cliente.close()

## Interpretação
1. Compare atrasos regionais no gráfico 01.
2. Verifique oscilações diárias no gráfico 02.
3. Observe adiantamentos no gráfico 03.
4. Avalie associação atraso/avaliação no gráfico 04, sem concluir causalidade.

Os dados são sintéticos; valide decisões com informações reais.